In [0]:
%pip install kagglehub

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import kagglehub
import os
# Download latest version
os.environ['KAGGLEHUB_CACHE'] = '/Volumes/workspace/gjain/project_health'
path = kagglehub.dataset_download("uciml/breast-cancer-wisconsin-data")

print("Path to dataset files:", path) 

Path to dataset files: /Volumes/workspace/gjain/project_health/datasets/uciml/breast-cancer-wisconsin-data/versions/2


In [0]:
# list the files
!ls "/Volumes/workspace/gjain/project_health/datasets/uciml/breast-cancer-wisconsin-data/versions/2"

data.csv


In [0]:
raw = spark.read.csv("/Volumes/workspace/gjain/project_health/datasets/uciml/breast-cancer-wisconsin-data/versions/2/data.csv", header=True, inferSchema=True)
print(raw.count())
raw.show(5)

569
+--------+---------+-----------+------------+--------------+---------+---------------+----------------+--------------+-------------------+-------------+----------------------+---------+----------+------------+-------+-------------+--------------+------------+-----------------+-----------+--------------------+------------+-------------+---------------+----------+----------------+-----------------+---------------+--------------------+--------------+-----------------------+----+
|      id|diagnosis|radius_mean|texture_mean|perimeter_mean|area_mean|smoothness_mean|compactness_mean|concavity_mean|concave points_mean|symmetry_mean|fractal_dimension_mean|radius_se|texture_se|perimeter_se|area_se|smoothness_se|compactness_se|concavity_se|concave points_se|symmetry_se|fractal_dimension_se|radius_worst|texture_worst|perimeter_worst|area_worst|smoothness_worst|compactness_worst|concavity_worst|concave points_worst|symmetry_worst|fractal_dimension_worst|_c32|
+--------+---------+-----------+--

In [0]:
"""
Naive Bayes classification example (scikit-learn).

- Uses GaussianNB (good default for continuous numeric features).
- Demonstrates train/test split, training, prediction, and evaluation.

Install deps:
  pip install scikit-learn

Run:
  python naive_bayes_classifier.py
"""

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


# 1) Load a sample binary classification dataset
df = raw.toPandas()

X = df.drop(columns=["diagnosis","id","_c32"])   # all columns except target
y = df["diagnosis"]

# 2) Split data
X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)
# 3) Train Naive Bayes model
model = GaussianNB()
model.fit(X_train, y_train)

# 4) Predict
y_pred = model.predict(X_test)

# 5) Evaluate
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.4f}\n")

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred), "\n")

print("Classification report:")
print(classification_report(y_test, y_pred))

(455, 30) (114, 30) (455,) (114,)
Accuracy: 0.9386

Confusion matrix:
[[72  0]
 [ 7 35]] 

Classification report:
              precision    recall  f1-score   support

           B       0.91      1.00      0.95        72
           M       1.00      0.83      0.91        42

    accuracy                           0.94       114
   macro avg       0.96      0.92      0.93       114
weighted avg       0.94      0.94      0.94       114



In [0]:
"""
step-by-step:
1) Parse CSV (expects header)
2) Encode diagnosis: M->1, B->0
3) Train: compute priors, per-class per-feature mean/variance
4) Predict: use log priors + sum log Gaussian pdfs
5) Evaluate accuracy

Assumptions:
- Input is a CSV file exported from your table, with columns including:
  id, diagnosis, <many numeric features>, and maybe an extra _c32 column.
- We ignore 'id' and any non-numeric columns besides diagnosis.
- Missing values like "NULL" are skipped by dropping that row (simple approach).
"""

import csv
import math
import random
import sys


# ----------------------------
# Step 1: Read CSV + basic cleaning
# ----------------------------
def is_float(s: str) -> bool:
    try:
        float(s)
        return True
    except Exception:
        return False


def load_dataset_csv(path: str):
    """
    Returns:
      X: list[list[float]]
      y: list[int] where M=1, B=0
      feature_names: list[str]
    Drops rows with any missing/non-numeric feature values.
    """
    with open(path, "r", newline="") as f:
        reader = csv.reader(f)
        header = next(reader)

        # Find diagnosis column; treat id as ignorable if present
        diag_idx = None
        for i, name in enumerate(header):
            if name.strip().lower() == "diagnosis":
                diag_idx = i
                break
        if diag_idx is None:
            raise ValueError("No 'diagnosis' column found.")

        X = []
        y = []

        # Decide which columns are numeric features:
        # Use the first data row to detect numeric columns (excluding id/diagnosis).
        first_row = next(reader)
        rows = [first_row] + list(reader)

        ignore_cols = set()
        for i, name in enumerate(header):
            lname = name.strip().lower()
            if lname in ("id", "diagnosis","_c32"):
                ignore_cols.add(i)

        feature_cols = []
        for i in range(len(header)):
            if i in ignore_cols:
                continue
            # If first row value is float-like and not NULL, treat as numeric feature
            v = first_row[i].strip()
            if v.upper() != "NULL" and is_float(v):
                feature_cols.append(i)

        feature_names = [header[i] for i in feature_cols]

        # Parse all rows
        for row in rows:
            d = row[diag_idx].strip()
            if d not in ("M", "B"):
                continue
            label = 1 if d == "M" else 0

            feats = []
            ok = True
            for i in feature_cols:
                v = row[i].strip()
                if v.upper() == "NULL" or (not is_float(v)):
                    ok = False
                    break
                feats.append(float(v))

            if ok:
                X.append(feats)
                y.append(label)

    return X, y, feature_names


# ----------------------------
# Step 2: Train/test split
# ----------------------------
def train_test_split(X, y, test_ratio=0.2, seed=42):
    idx = list(range(len(X)))
    random.Random(seed).shuffle(idx)

    n_test = int(len(X) * test_ratio)
    test_idx = idx[:n_test]
    train_idx = idx[n_test:]

    X_train = [X[i] for i in train_idx]
    y_train = [y[i] for i in train_idx]
    X_test = [X[i] for i in test_idx]
    y_test = [y[i] for i in test_idx]
    return X_train, X_test, y_train, y_test


# ----------------------------
# Step 3: Gaussian NB training (compute priors, means, variances)
# ----------------------------
def mean(xs):
    return sum(xs) / len(xs) if xs else 0.0


def variance(xs, mu):
    # population variance (divide by N). For NB either is fine; keep simple.
    if not xs:
        return 0.0
    return sum((x - mu) ** 2 for x in xs) / len(xs)


def train_gaussian_nb(X_train, y_train, eps=1e-9):
    """
    Returns model:
      classes = [0, 1]
      prior[c]
      mean[c][j]
      var[c][j]  (with small eps added to avoid divide-by-zero)
    """
    classes = sorted(list(set(y_train)))
    n_features = len(X_train[0])

    # collect values per class per feature
    by_class = {c: [[] for _ in range(n_features)] for c in classes}
    class_counts = {c: 0 for c in classes}

    for x, y in zip(X_train, y_train):
        class_counts[y] += 1
        for j in range(n_features):
            by_class[y][j].append(x[j])

    n_total = len(X_train)
    prior = {c: class_counts[c] / n_total for c in classes}

    mu = {c: [0.0] * n_features for c in classes}
    var = {c: [0.0] * n_features for c in classes}

    for c in classes:
        for j in range(n_features):
            m = mean(by_class[c][j])
            v = variance(by_class[c][j], m) + eps  # eps stabilizes
            mu[c][j] = m
            var[c][j] = v

    return {"classes": classes, "prior": prior, "mean": mu, "var": var}


# ----------------------------
# Step 4: Gaussian log-likelihood + prediction
# ----------------------------
def log_gaussian_pdf(x, mu, var):
    # log N(x | mu, var)
    # = -0.5*log(2*pi*var) - (x-mu)^2/(2*var)
    return -0.5 * math.log(2.0 * math.pi * var) - ((x - mu) ** 2) / (2.0 * var)


def predict_one(model, x):
    classes = model["classes"]
    prior = model["prior"]
    mu = model["mean"]
    var = model["var"]

    scores = {}
    for c in classes:
        s = math.log(prior[c])
        for j, xj in enumerate(x):
            s += log_gaussian_pdf(xj, mu[c][j], var[c][j])
        scores[c] = s

    best = max(scores, key=scores.get)
    return best, scores


def predict(model, X):
    return [predict_one(model, x)[0] for x in X]


# ----------------------------
# Step 5: Evaluate
# ----------------------------
def accuracy(y_true, y_pred):
    correct = 0
    for yt, yp in zip(y_true, y_pred):
        if yt == yp:
            correct += 1
    return correct / len(y_true) if y_true else 0.0


def confusion_matrix_binary(y_true, y_pred):
    # labels: 1=M, 0=B
    tp = fp = tn = fn = 0
    for yt, yp in zip(y_true, y_pred):
        if yt == 1 and yp == 1:
            tp += 1
        elif yt == 0 and yp == 1:
            fp += 1
        elif yt == 0 and yp == 0:
            tn += 1
        elif yt == 1 and yp == 0:
            fn += 1
    return tp, fp, tn, fn


def main():
    if len(sys.argv) < 2:
        print("Usage: python gaussian_naive_bayes_from_scratch.py <data.csv>")
        sys.exit(1)

    
    X, y, feature_names = load_dataset_csv("/Volumes/workspace/gjain/project_health/datasets/uciml/breast-cancer-wisconsin-data/versions/2/data.csv")

    print(f"Loaded rows: {len(X)}")
    print(f"Number of features used: {len(feature_names)}")
    print("First 5 feature names:", feature_names[:5])

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_ratio=0.2, seed=42)

    model = train_gaussian_nb(X_train, y_train, eps=1e-9)
    y_pred = predict(model, X_test)

    acc = accuracy(y_test, y_pred)
    tp, fp, tn, fn = confusion_matrix_binary(y_test, y_pred)

    print("\nResults:")
    print(f"Accuracy: {acc:.4f}")
    print(f"Confusion matrix (tp, fp, tn, fn): {tp}, {fp}, {tn}, {fn}")

    # Optional: show a few predictions
    print("\nSample predictions (first 10):")
    for i in range(min(10, len(X_test))):
        pred_label = "M" if y_pred[i] == 1 else "B"
        true_label = "M" if y_test[i] == 1 else "B"
        print(f"  true={true_label} pred={pred_label}")

    # Optional: predict a single row and inspect log-scores
    # yhat, scores = predict_one(model, X_test[0])
    # print(scores)


if __name__ == "__main__":
    main()

---------------------------------------------------------------------------
IndexError                                Traceback (most recent call last)
File <command-4766031775778411>, line 261
    255     # Optional: predict a single row and inspect log-scores
    256     # yhat, scores = predict_one(model, X_test[0])
    257     # print(scores)
    260 if __name__ == "__main__":
--> 261     main()

File <command-4766031775778411>, line 230, in main()
    226 def main():
--> 230     X, y, feature_names = load_dataset_csv("/Volumes/workspace/gjain/project_health/datasets/uciml/breast-cancer-wisconsin-data/versions/2/data.csv")
    232     print(f"Loaded rows: {len(X)}")
    233     print(f"Number of features used: {len(feature_names)}")

File <command-4766031775778411>, line 73, in load_dataset_csv(path)
     71     continue
     72 # If first row value is float-like and not NULL, treat as numeric feature
---> 73 v = first_row[i].strip()
     74 if v.upper() != "NULL" and is_float(v):
